In [0]:
df_order_items = spark.table("ecommerce_dev.bronze.order_items")

df_order_items.printSchema()
df_order_items.limit(5).display()
print(f"Row count: {df_order_items.count()}")

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_rescued_data,_ingested_at,_source_file,_source_table
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,null,2026-08-04T00:55:16.101Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv,order_items
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,null,2026-08-04T00:55:16.101Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv,order_items
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,null,2026-08-04T00:55:16.101Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv,order_items
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,null,2026-08-04T00:55:16.101Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv,order_items
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,null,2026-08-04T00:55:16.101Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv,order_items


Row count: 112650


In [0]:
from pyspark.sql.functions import *

# Nulls per column
df_order_items.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_order_items.columns]
).display()

# Full-row duplicates
print("Full duplicate rows:", df_order_items.count() - df_order_items.dropDuplicates().count())

# composite key check — order_id + order_item_id should be unique together
print("Duplicate (order_id, order_item_id):",
      df_order_items.count() - df_order_items.dropDuplicates(["order_id", "order_item_id"]).count())

# check _rescued_data
df_order_items.select("_rescued_data").filter(col("_rescued_data").isNotNull()).display()

# numeric sanity — negative or zero prices/freight would be suspicious
df_order_items.select(
    min("price").alias("min_price"), max("price").alias("max_price"),
    min("freight_value").alias("min_freight"), max("freight_value").alias("max_freight")
).display()

print("Rows with price <= 0:", df_order_items.filter(col("price") <= 0).count())
print("Rows with freight_value < 0:", df_order_items.filter(col("freight_value") < 0).count())

# check FK integrity against orders/products/sellers we already cleaned
print("order_items with order_id not in orders:",
      df_order_items.join(spark.table("ecommerce_dev.silver.orders"), "order_id", "left_anti").count())

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_rescued_data,_ingested_at,_source_file,_source_table
0,0,0,0,0,0,0,112650,0,0,0


Full duplicate rows: 0
Duplicate (order_id, order_item_id): 0


_rescued_data


min_price,max_price,min_freight,max_freight
0.85,6735.0,0.0,409.68


Rows with price <= 0: 0
Rows with freight_value < 0: 0
order_items with order_id not in orders: 0


In [0]:
print("order_items with seller_id not in sellers:",
      df_order_items.join(spark.table("ecommerce_dev.silver.sellers"), "seller_id", "left_anti").count())

order_items with seller_id not in sellers: 0


In [0]:
from pyspark.sql.functions import *

df_silver_order_items = (
    df_order_items
    .drop("_rescued_data", "_source_file", "_source_table")
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
)

df_silver_order_items.limit(5).display()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,bronze_ingested_at
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-08-04T00:55:16.101Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-08-04T00:55:16.101Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-08-04T00:55:16.101Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-08-04T00:55:16.101Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-08-04T00:55:16.101Z


In [0]:
(df_silver_order_items.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("ecommerce_dev.silver.order_items"))

In [0]:
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ALTER COLUMN order_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ALTER COLUMN order_item_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ADD CONSTRAINT pk_order_items PRIMARY KEY (order_id, order_item_id)
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ADD CONSTRAINT fk_order_items_order FOREIGN KEY (order_id)
    REFERENCES ecommerce_dev.silver.orders (order_id)
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ADD CONSTRAINT fk_order_items_seller FOREIGN KEY (seller_id)
    REFERENCES ecommerce_dev.silver.sellers (seller_id)
""")

DataFrame[]

In [0]:
spark.sql("""
    COMMENT ON TABLE ecommerce_dev.silver.order_items IS
    'Cleaned order line items. Composite PK: (order_id, order_item_id). FKs: order_id -> silver.orders, seller_id -> silver.sellers. product_id FK deferred until silver.products exists. Verified positive price/freight, no dupes, 100% FK integrity to orders and sellers.'
""")

DataFrame[]

In [0]:
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_items
    ADD CONSTRAINT fk_order_items_product FOREIGN KEY (product_id)
    REFERENCES ecommerce_dev.silver.products (product_id)
""")

DataFrame[]